# GymRAVANA public pose retraining

This executed notebook retrains the separate five-class **pose identity** prototype using the manually downloaded Yoga-107 archive. It uses MediaPipe landmark features and duplicate-grouped evaluation. It cannot assess form correctness, safety, therapy needs, or progression readiness.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ai.pose.public_workflow import (
    evaluate_public_candidates,
    export_public_prototype,
    prepare_public_features,
)

DATASET_DIR = PROJECT_ROOT / 'ai/data/pose_public'
MODEL_PATH = PROJECT_ROOT / 'ai/models/pose_landmarker_lite.task'
CSV_PATH = PROJECT_ROOT / 'ai/data/pose_public_features.csv'
METADATA_PATH = PROJECT_ROOT / 'ai/data/pose_public_features.metadata.json'
REPORT_PATH = PROJECT_ROOT / 'ai/artifacts/pose_public_model_selection.json'
ARTIFACT_DIR = PROJECT_ROOT / 'ai/artifacts'

## 1. Verify images and extract real landmarks

Unreadable images, label conflicts and images where MediaPipe cannot detect a pose are recorded and excluded. No landmarks or labels are fabricated.

In [2]:
metadata = prepare_public_features(DATASET_DIR, MODEL_PATH, CSV_PATH, METADATA_PATH)
preparation_summary = {
    'valid_landmark_rows': metadata['row_count'],
    'class_counts': metadata['class_counts'],
    'visual_groups': metadata['visual_group_count'],
    'duplicate_groups': metadata['duplicate_group_count'],
    'cross_class_near_duplicate_pairs': len(metadata['cross_class_near_duplicate_pairs']),
    'image_audit_failures': len(metadata['image_audit_failures']),
    'landmark_failures': len(metadata['landmark_failures']),
}
print(json.dumps(preparation_summary, indent=2))
assert metadata['class_count'] == 5
assert metadata['row_count'] >= 250

C:\Users\anjan\OneDrive\Documents\Gym-RAVNA\ravana-app\.venv\Lib\site-packages\PIL\Image.py:1136: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


{
  "valid_landmark_rows": 271,
  "class_counts": {
    "balasana": 64,
    "mayurasana": 50,
    "salamba_sirsasana": 52,
    "urdhva_dhanurasana": 56,
    "virasana": 49
  },
  "visual_groups": 270,
  "duplicate_groups": 2,
  "cross_class_near_duplicate_pairs": 0,
  "image_audit_failures": 0,
  "landmark_failures": 29
}


## 2. Duplicate-grouped model selection and untouched holdout

Near-duplicate visual groups remain in one partition. This reduces obvious leakage, but it is not a substitute for participant-grouped evaluation because Yoga-107 supplies no participant IDs.

In [3]:
evaluation = evaluate_public_candidates(CSV_PATH, METADATA_PATH, REPORT_PATH)
candidate_table = pd.DataFrame([
    {
        'model': name,
        'development_mean_accuracy': result['mean_accuracy'],
        'development_mean_macro_f1': result['mean_macro_f1'],
    }
    for name, result in evaluation['candidates'].items()
]).sort_values('development_mean_macro_f1', ascending=False)
display(candidate_table)
print(json.dumps({
    'selected_model': evaluation['selected_model'],
    'development_rows': evaluation['development_rows'],
    'holdout_rows': evaluation['holdout_rows'],
    'holdout_accuracy': evaluation['holdout']['accuracy'],
    'holdout_macro_f1': evaluation['holdout']['macro_f1'],
    'evidence_gates': evaluation['evidence_gates'],
    'deployment_allowed': evaluation['deployment_allowed'],
}, indent=2))
assert evaluation['deployment_allowed'] is False

,model,development_mean_accuracy,development_mean_macro_f1
1,random_forest,0.894023,0.891147
0,logistic_regression,0.824832,0.820508
2,support_vector_machine,0.811111,0.808996


{
  "selected_model": "random_forest",
  "development_rows": 217,
  "holdout_rows": 54,
  "holdout_accuracy": 0.925926,
  "holdout_macro_f1": 0.921501,
  "evidence_gates": {
    "minimum_250_landmark_rows": true,
    "minimum_10_visual_groups": true,
    "participant_ids_available": false,
    "trainer_verified_labels": false,
    "local_camera_test_available": false,
    "form_correctness_target_available": false
  },
  "deployment_allowed": false
}


## 3. Explainability and prototype export

Permutation importance uses the untouched holdout with a development-only explanation model. The serialized prototype is then trained on all valid rows, fingerprinted, and kept disconnected from member-facing routes.

In [4]:
artifact = export_public_prototype(CSV_PATH, METADATA_PATH, REPORT_PATH, ARTIFACT_DIR)
importance = json.loads((ARTIFACT_DIR / 'pose_identity_public_prototype.feature_importance.json').read_text())
display(pd.DataFrame(importance).head(10))
print(json.dumps({
    'model_name': artifact['model_name'],
    'training_rows': artifact['training_rows'],
    'model_sha256': artifact['model_sha256'],
    'prototype_only': artifact['prototype_only'],
    'deployment_allowed': artifact['deployment_allowed'],
}, indent=2))
assert artifact['prototype_only'] is True
assert artifact['deployment_allowed'] is False

,feature,mean_importance,std_importance
0,torso_angle_deg,0.070019,0.025312
1,left_shoulder_angle,0.021052,0.012349
2,left_elbow_angle,0.020346,0.020447
3,left_hip_angle,0.016375,0.019248
4,right_knee_angle,0.014529,0.013831
5,left_knee_angle,0.013589,0.014303
6,right_hip_angle,0.009747,0.017027
7,right_elbow_angle,0.003822,0.012865
8,right_shoulder_angle,0.003469,0.018510
9,hip_slope_deg,0.000000,0.000000


{
  "model_name": "random_forest",
  "training_rows": 271,
  "model_sha256": "25756929202e18c9604e110e12e62222cc0d99ea2e8fc632634fd9bacfb04380",
  "prototype_only": true,
  "deployment_allowed": false
}


## Conclusion

This is a materially stronger undergraduate pose-identity experiment than the 15-image starter run. It is still not deployable because the web dataset lacks participant IDs, trainer-verified correctness labels and a test recorded under GymRAVANA's intended camera conditions.